In [1]:
import pygsti
import itertools
from pygsti.extras import ml
import numpy as np
from pygsti.circuits import Circuit
from pygsti.baseobjs import Label as L, Basis, QubitSpace
from pygsti.processors.processorspec import QubitProcessorSpec as QPS
from matplotlib import pyplot as plt
from pygsti.algorithms.randomcircuit import create_random_circuit as random_circuit
import tensorflow as tf
%load_ext autoreload
%autoreload 2

2026-02-02 12:39:32.230718: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
num_qubits = 4

gate_names = ['Gxpi2', 'Gypi2', 'Gcphase']

availability= {'Gcphase': [(0,1), (1,2), (2,3), (3,0)]}

qubit_labels = [i for i in range(num_qubits)]

pspec = QPS(num_qubits = num_qubits, qubit_labels=qubit_labels,
            gate_names=gate_names, availability=availability)


In [3]:
num_circs = 300
circuits = [random_circuit(pspec, np.random.randint(1, 30 + 1), sampler='edgegrab', samplerargs=[0.5]) for i in range (num_circs)]

In [4]:
def sample_errors_dict(scale_factor=1, crosstalk_scale_factor=1):

    error_rates_dict = {}
    
    def sample_local_oneQ_Hamiltonian_error():
        return scale_factor * 0.01 * 2 * (np.random.rand() - 0.5)
    
    def sample_local_oneQ_stochastic_error():
        return scale_factor * 0.001 * np.random.rand()
    
    def sample_neighbor_oneQ_Hamiltonian_error():
        return crosstalk_scale_factor * scale_factor * 0.005 * 2 * (np.random.rand() - 0.5)
    
    def sample_local_twoQ_Hamiltonian_error():
        return scale_factor * 0.01 * 2 * (np.random.rand() - 0.5)
    
    def sample_local_twoQ_stochastic_error():
        return scale_factor * 0.001  * np.random.rand()
    
    def sample_twoQ_Hamiltonian_ZZZZ_error():
        return crosstalk_scale_factor * scale_factor * 0.02  * np.random.rand()
    
    error_rates_dict[L('Gypi2',0)] = {('H','YIII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gypi2',1)] = {('H','IYII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gypi2',2)] = {('H','IIYI:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gypi2',3)] = {('H','IIIY:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    
    error_rates_dict[L('Gxpi2',0)] = {('H','XIII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gxpi2',1)] = {('H','IXII:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gxpi2',2)] = {('H','IIXI:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    error_rates_dict[L('Gxpi2',3)] = {('H','IIIX:0,1,2,3'): sample_local_oneQ_Hamiltonian_error()}
    
    error_rates_dict[L('Gypi2',0)].update({('S','YIII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gypi2',1)].update({('S','IYII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gypi2',2)].update({('S','IIYI:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gypi2',3)].update({('S','IIIY:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    
    error_rates_dict[L('Gxpi2',0)].update({('S','XIII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gxpi2',1)].update({('S','IXII:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gxpi2',2)].update({('S','IIXI:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    error_rates_dict[L('Gxpi2',3)].update({('S','IIIX:0,1,2,3'): sample_local_oneQ_stochastic_error()})
    
    error_rates_dict[L('Gxpi2',0)].update({('H','IXII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',0)].update({('H','IIIX:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',1)].update({('H','XIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',1)].update({('H','IIXI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',2)].update({('H','IXII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',2)].update({('H','IIIX:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',3)].update({('H','IIXI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gxpi2',3)].update({('H','XIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    
    error_rates_dict[L('Gypi2',0)].update({('H','IYII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',0)].update({('H','IIIY:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',1)].update({('H','YIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',1)].update({('H','IIYI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',2)].update({('H','IYII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',2)].update({('H','IIIY:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',3)].update({('H','IIYI:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    error_rates_dict[L('Gypi2',3)].update({('H','YIII:0,1,2,3'): sample_neighbor_oneQ_Hamiltonian_error()})
    
    error_rates_dict[L('Gcphase',(0,1))] = {('H','ZZII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(0,1))].update({('H','ZIII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(0,1))].update({('H','IZII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(1,2))] = {('H','IZZI:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(1,2))].update({('H','IZII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('H','IIZI:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(2,3))] = {('H','IIZZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(2,3))].update({('H','IIIZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('H','IIZI:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(3,0))] = {('H','ZIIZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()}
    error_rates_dict[L('Gcphase',(3,0))].update({('H','ZIII:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('H','IIIZ:0,1,2,3'): sample_local_twoQ_Hamiltonian_error()})
    
    error_rates_dict[L('Gcphase',(0,1))].update({('S','ZZII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(0,1))].update({('S','ZIII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(0,1))].update({('S','IZII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('S','IZZI:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('S','IZII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('S','IIZI:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('S','IIZZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('S','IIIZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('S','IIZI:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('S','ZIIZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('S','ZIII:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('S','IIIZ:0,1,2,3'): sample_local_twoQ_stochastic_error()})
    
    error_rates_dict[L('Gcphase',(0,1))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})
    error_rates_dict[L('Gcphase',(1,2))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})
    error_rates_dict[L('Gcphase',(2,3))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})
    error_rates_dict[L('Gcphase',(3,0))].update({('H','ZZZZ:0,1,2,3'): sample_twoQ_Hamiltonian_ZZZZ_error()})

    return error_rates_dict

In [5]:
error_rates_dict = sample_errors_dict()
error_model = pygsti.models.create_cloud_crosstalk_model(pspec, lindblad_error_coeffs=error_rates_dict,
                                                         errcomp_type="errorgens")

In [6]:
nbit_strings = [''.join(p) for p in itertools.product('01', repeat=num_qubits)]
simulated_probabilities = []
for i, circuit in enumerate(circuits):
    probs_dict =  error_model.probabilities(circuit)
    simulated_probabilities.append([probs_dict[bs] for bs in nbit_strings])
simulated_probabilities = np.array(simulated_probabilities)

In [7]:
# We're going to create a neural network with all the correct error generators, so
# this extracts them from the error_rates_dict.
modelled_error_generators  = []
for gate_error_rates in error_rates_dict.values():
    for error_label in gate_error_rates.keys():
        error_gen = (error_label[0], (error_label[1].split(':')[0],))
        if error_gen not in modelled_error_generators:
            modelled_error_generators.append(error_gen)
print(modelled_error_generators)
print(len(modelled_error_generators))

[('H', ('YIII',)), ('S', ('YIII',)), ('H', ('IYII',)), ('H', ('IIIY',)), ('S', ('IYII',)), ('H', ('IIYI',)), ('S', ('IIYI',)), ('S', ('IIIY',)), ('H', ('XIII',)), ('S', ('XIII',)), ('H', ('IXII',)), ('H', ('IIIX',)), ('S', ('IXII',)), ('H', ('IIXI',)), ('S', ('IIXI',)), ('S', ('IIIX',)), ('H', ('ZZII',)), ('H', ('ZIII',)), ('H', ('IZII',)), ('S', ('ZZII',)), ('S', ('ZIII',)), ('S', ('IZII',)), ('H', ('ZZZZ',)), ('H', ('IZZI',)), ('H', ('IIZI',)), ('S', ('IZZI',)), ('S', ('IIZI',)), ('H', ('IIZZ',)), ('H', ('IIIZ',)), ('S', ('IIZZ',)), ('S', ('IIIZ',)), ('H', ('ZIIZ',)), ('S', ('ZIIZ',))]
33


In [10]:
 set(np.array([2,2,3]))

{2, 3}

In [22]:
tensors = ml.encoding.error_generator_tensors(circuits, modelled_error_generators, pspec, alpha_representation='concise')
indices = tensors['indices'] # Not needed by the QPANN but are computed to compute the alphas and so still returned
signs = tensors['signs']  # Not needed by the QPANN but are computed to compute the alphas and so still returned
probabilities = tensors['probabilities'] 
alphas = tensors['alphas']

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌    | 293/300 [2:36:21<03:44, 32.02s/it]


KeyboardInterrupt: 

In [ ]:
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
x = [circuits_tensor, alphas, probabilities]
qpann = ml.qpanns.QPANN(encoder.length, modelled_error_generators, snipper, probability_computation='concise')

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
early_stopping = tf.keras.callbacks.EarlyStopping(
                monitor='val_R2Score',  # Monitor the validation loss
                patience=20,         # Number of epochs with no improvement after which training will be stopped
                restore_best_weights=True,  # Restore model weights from the epoch with the best validation loss
                verbose=1            # Verbosity mode, 1 = progress messages
            )

qpann.compile(optimizer, loss=tf.keras.losses.MSE, metrics=['R2Score', 'mae'])
qpann.summary()
# Fit the model using training data and include validation data
history = qpann.fit(
    x, 
    simulated_probabilities, 
    epochs=100,
    #validation_data=(x_new['validate'], y_split[sfs][nm_ind][num_shots]['validate']),  # Include validation data
    callbacks=[early_stopping], 
    verbose=1, 
    batch_size=100, 
    validation_batch_size=100,
    shuffle=True
)

fig = plt.figure(figsize=(7,4))
plt.plot(history.history['loss'], label='loss')